# 04 – Hypothesis Testing I

You've run a product experiment. The new website design showed a 3% higher conversion rate than the old one. But was that a real improvement — or just random variation in who happened to visit the site that week?

You've changed a drug dosage and the test group showed better outcomes. Is the drug actually working, or could those results have happened by chance even if the drug did nothing?

**Hypothesis testing** is the formal procedure for answering these questions. It gives you a structured way to decide: is what I observed real, or is it noise?

This notebook covers:
- The framework: null hypothesis, alternative hypothesis, test statistic
- The p-value — what it really means (and what it doesn't)
- Type I and Type II errors — the two ways tests can be wrong
- The one-sample t-test — the most common starting point

## 1) The Framework — How Hypothesis Testing Works

Every hypothesis test follows the same structure:

**Step 1:** State the null hypothesis (H₀)
- The "nothing is happening" claim
- The claim you try to **disprove**
- Example: "This drug has no effect." / "Conversion rates are the same."

**Step 2:** State the alternative hypothesis (H₁)
- What you believe is actually true
- Example: "This drug improves outcomes." / "The new design converts better."

**Step 3:** Set the significance level (α)
- How much risk of a false positive are you willing to accept?
- Standard: α = 0.05 (5%)

**Step 4:** Compute the test statistic
- How many standard errors away from the null is your observed result?

**Step 5:** Compute the p-value
- How likely is your result (or more extreme) if H₀ were true?

**Step 6:** Decision
- If p-value < α → reject H₀ (result is "statistically significant")
- If p-value ≥ α → fail to reject H₀ (not enough evidence)

**Crucial:** "Fail to reject" ≠ "H₀ is true." It just means you don't have enough evidence to reject it.

## 2) The p-value — The Most Misunderstood Number in Statistics

**What the p-value IS:**
> The probability of observing a result as extreme as (or more extreme than) what you observed, **assuming the null hypothesis is true**.

**What the p-value IS NOT:**
- NOT the probability that H₀ is true
- NOT the probability that your result is due to chance
- NOT the probability that your finding will replicate

A small p-value (< 0.05) means: "If H₀ were true, what I observed would be very unlikely."

That's evidence against H₀. It's not proof that H₁ is true.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Visualise what the p-value IS
# H₀: mean = 70 (null hypothesis about exam scores)
# Observed: sample mean = 75.8 from n=30 students, s=12

mu_0 = 70      # null hypothesis value
x_bar = 75.8   # observed sample mean
s = 12.0       # sample std
n = 30         # sample size
se = s / np.sqrt(n)

t_stat = (x_bar - mu_0) / se
p_val  = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))   # two-tailed

print(f"Null hypothesis mean : μ₀ = {mu_0}")
print(f"Observed sample mean : x̄ = {x_bar}")
print(f"Standard error       : SE = {se:.3f}")
print(f"t-statistic          : t = {t_stat:.3f}")
print(f"p-value (two-tailed) : p = {p_val:.4f}")

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(-5, 5, 1000)
t_dist = stats.t.pdf(x, df=n-1)
ax.plot(x, t_dist, color="steelblue", linewidth=2.5)

# Shade the p-value regions (both tails for two-tailed test)
ax.fill_between(x, t_dist, where=(x >= abs(t_stat)), alpha=0.4, color="firebrick",
                label=f"p-value/2 = {p_val/2:.4f} (right tail)")
ax.fill_between(x, t_dist, where=(x <= -abs(t_stat)), alpha=0.4, color="firebrick",
                label=f"p-value/2 = {p_val/2:.4f} (left tail)")

ax.axvline(t_stat,  color="firebrick", linewidth=2.5, linestyle="--", label=f"Observed t = {t_stat:.2f}")
ax.axvline(-t_stat, color="firebrick", linewidth=2.5, linestyle="--")
ax.axvline(1.96, color="gray", linewidth=1.5, linestyle=":", label="Critical value ±1.96 (α=0.05)")
ax.axvline(-1.96, color="gray", linewidth=1.5, linestyle=":")
ax.axvline(0, color="black", linewidth=1)
ax.set_title(f"t-distribution under H₀: μ={mu_0}\np-value = {p_val:.4f}  |  α = 0.05", fontweight="bold")
ax.set_xlabel("t-statistic")
ax.set_ylabel("Probability Density")
ax.legend(fontsize=9)
ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("/home/claude/06_stats/pvalue_plot.png", dpi=120)
plt.close()
print("p-value visualisation saved.")

**Reading the chart:**

The t-distribution curve shows the probability of each t-statistic value **if H₀ were true** (if the true mean really were 70).

The red shaded area is the p-value — the probability of getting a t-statistic as extreme as ours (or more extreme) by chance alone.

If the red area (p-value) is smaller than α (0.05), the observed result is too unlikely to be explained by chance under H₀. We reject H₀.

If p-value > α, the result could plausibly have occurred by chance. We don't have enough evidence to reject H₀.

## 3) Type I and Type II Errors

No test is perfect. There are exactly two ways a hypothesis test can be wrong:

```
                    │  H₀ is TRUE        │  H₀ is FALSE
────────────────────┼────────────────────┼──────────────────
Reject H₀           │  Type I Error (FP) │  Correct ✓
                    │  "False alarm"     │  ("Power")
────────────────────┼────────────────────┼──────────────────
Fail to Reject H₀   │  Correct ✓         │  Type II Error (FN)
                    │                    │  "Missed detection"
```

- **Type I Error (False Positive):** You reject H₀ when it was actually true. Rate = α (significance level)
- **Type II Error (False Negative):** You fail to reject H₀ when it was actually false. Rate = β

**Power = 1 − β** — the probability of correctly detecting a real effect.

**The tradeoff:** Making α smaller (more conservative) reduces Type I errors but increases Type II errors. You can't simultaneously minimise both without increasing sample size.

In [ ]:
# Visualise the tradeoff between Type I and Type II errors
fig, ax = plt.subplots(figsize=(11, 5))

mu_0   = 0       # null hypothesis: no effect
mu_1   = 1.5     # alternative: effect exists (true difference)
se_val = 1.0     # standard error
alpha  = 0.05
critical_val = stats.norm.ppf(1 - alpha/2)   # two-tailed 95%

x = np.linspace(-4, 6, 1000)
h0_dist = stats.norm.pdf(x, mu_0, se_val)
h1_dist = stats.norm.pdf(x, mu_1, se_val)

ax.plot(x, h0_dist, color="steelblue",  linewidth=2.5, label="H₀ distribution (no effect)")
ax.plot(x, h1_dist, color="seagreen",   linewidth=2.5, label="H₁ distribution (real effect)")
ax.axvline( critical_val, color="gray", linestyle="--", linewidth=1.5)
ax.axvline(-critical_val, color="gray", linestyle="--", linewidth=1.5, label=f"Critical values ±{critical_val:.2f}")

# Type I error — rejecting a true H₀ (right tail of H₀)
ax.fill_between(x, h0_dist, where=(x >= critical_val), alpha=0.5, color="firebrick",
                label=f"Type I error (α = {alpha})")
# Type II error — failing to reject a false H₀ (left part of H₁)
ax.fill_between(x, h1_dist, where=(x <= critical_val), alpha=0.4, color="darkorange",
                label="Type II error (β)")

# Power — correctly rejecting H₀
power = 1 - stats.norm.cdf(critical_val, mu_1, se_val)
ax.fill_between(x, h1_dist, where=(x >= critical_val), alpha=0.4, color="seagreen",
                label=f"Power = {power:.2f}")

ax.set_title("Type I Error, Type II Error, and Power", fontsize=13, fontweight="bold")
ax.set_xlabel("Test Statistic")
ax.set_ylabel("Probability Density")
ax.legend(fontsize=9)
ax.grid(linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/06_stats/errors_plot.png", dpi=120)
plt.close()
print(f"Error visualisation saved.")
print(f"Power = {power:.4f}  (probability of correctly detecting the effect)")

**Key insight:** The two distributions overlap. The overlap is where errors happen.

- The **red area** (under H₀ curve, beyond critical value) — where you'd reject H₀ even though H₀ is true → Type I error
- The **orange area** (under H₁ curve, before critical value) — where you'd fail to reject H₀ even though H₁ is true → Type II error
- The **green area** (under H₁ curve, beyond critical value) — where you correctly reject H₀ → Power

If the two distributions are further apart (larger effect), the orange area shrinks and power increases. If sample size increases, both distributions become narrower and the overlap decreases.

## 4) The One-Sample t-Test

**When to use:** You have a sample and you want to test whether the population mean equals some specific hypothesised value.

**Example:** A coffee machine is supposed to dispense 250 ml per cup. You take 25 samples and get a mean of 247 ml. Is the machine miscalibrated?

H₀: μ = 250 ml (machine is correctly calibrated)  
H₁: μ ≠ 250 ml (machine is miscalibrated) — two-tailed test

In [ ]:
# One-sample t-test — coffee machine example
rng = np.random.default_rng(7)

# Simulate 25 measurements from a slightly miscalibrated machine (true mean = 247)
true_mean = 247
sample_measurements = rng.normal(loc=true_mean, scale=8, size=25)

print("=== Coffee Machine Calibration Test ===")
print()
print(f"H₀: machine dispenses μ = 250 ml (correctly calibrated)")
print(f"H₁: machine dispenses μ ≠ 250 ml (miscalibrated)")
print(f"α  = 0.05 (5% significance level)")
print()
print(f"Sample size  : {len(sample_measurements)}")
print(f"Sample mean  : {sample_measurements.mean():.2f} ml")
print(f"Sample std   : {sample_measurements.std(ddof=1):.2f} ml")
print()

# Using scipy.stats.ttest_1samp — the clean way
t_stat, p_val = stats.ttest_1samp(sample_measurements, popmean=250)

print(f"t-statistic  : {t_stat:.4f}")
print(f"p-value      : {p_val:.4f}")
print()
if p_val < 0.05:
    print("Decision: REJECT H₀ — statistically significant evidence of miscalibration")
else:
    print("Decision: FAIL TO REJECT H₀ — no statistically significant evidence of miscalibration")

# Confidence interval for the true mean
se = sample_measurements.std(ddof=1) / np.sqrt(len(sample_measurements))
t_crit = stats.t.ppf(0.975, df=len(sample_measurements)-1)
ci_lo = sample_measurements.mean() - t_crit * se
ci_hi = sample_measurements.mean() + t_crit * se
print()
print(f"95% CI for true mean: ({ci_lo:.2f}, {ci_hi:.2f}) ml")
print(f"Hypothesised value (250) in CI? {ci_lo <= 250 <= ci_hi}")

**Notice:** The confidence interval and the hypothesis test always agree for the same α.

If 250 is outside the 95% CI → p-value < 0.05 → reject H₀.  
If 250 is inside the 95% CI → p-value ≥ 0.05 → fail to reject H₀.

They're two ways of saying the same thing.

## 5) One-Tailed vs Two-Tailed Tests

**Two-tailed:** You care about differences in both directions.
- H₁: μ ≠ 250 (could be too high or too low)
- Divide α between both tails: each tail gets α/2 = 0.025

**One-tailed (right):** You only care if the value is higher.
- H₁: μ > 250
- All of α goes in the right tail

**One-tailed (left):** You only care if the value is lower.
- H₁: μ < 250
- All of α goes in the left tail

**When to use one-tailed:** Only when it would be irrelevant (or even good) if the effect went in the other direction. In practice, two-tailed is usually safer and more honest.

In [ ]:
# One-tailed vs two-tailed comparison
# Hypothesis: a new teaching method improves scores (one-tailed)
# vs just changes scores (two-tailed)

rng = np.random.default_rng(12)
baseline_mean = 68
new_scores = rng.normal(loc=71, scale=15, size=40)   # small improvement

print("=== One-Tailed vs Two-Tailed ===")
print()
print(f"Baseline mean : {baseline_mean}")
print(f"Sample mean   : {new_scores.mean():.2f}")
print(f"Sample std    : {new_scores.std(ddof=1):.2f}")
print(f"n             : {len(new_scores)}")
print()

# Two-tailed: H₁: μ ≠ 68
t_stat, p_two = stats.ttest_1samp(new_scores, popmean=baseline_mean)
# One-tailed (right): H₁: μ > 68
p_one_right = p_two / 2 if t_stat > 0 else 1 - p_two / 2

print(f"t-statistic          : {t_stat:.4f}")
print(f"p-value (two-tailed) : {p_two:.4f}  → {'Reject H₀' if p_two < 0.05 else 'Fail to reject H₀'}")
print(f"p-value (one-tailed) : {p_one_right:.4f}  → {'Reject H₀' if p_one_right < 0.05 else 'Fail to reject H₀'}")
print()
print("One-tailed p = two-tailed p / 2 (when t is in the expected direction)")
print("More powerful, but only valid if you committed to the direction BEFORE seeing data.")

**Warning:** Never switch from two-tailed to one-tailed after seeing your data to get a significant result. That's p-hacking — a form of data manipulation. You must commit to the test type and direction before collecting data.

## 6) Statistical Significance vs Practical Significance

Statistical significance tells you: *Is this effect real?*

Practical significance tells you: *Is this effect large enough to matter?*

With large sample sizes, even trivially tiny differences become statistically significant. With small samples, large real effects can fail to reach significance.

**Cohen's d** measures effect size for mean comparisons:

```
d = (x̄ - μ₀) / s
```

- d ≈ 0.2 → small effect
- d ≈ 0.5 → medium effect
- d ≈ 0.8 → large effect

In [ ]:
# Statistical vs practical significance
rng = np.random.default_rng(99)

print("=== Sample Size Effect on Statistical Significance ===")
print()
print(f"True effect: website A converts at 10.0%, website B at 10.1%")
print(f"(A 0.1 percentage point difference — practically meaningless)")
print()

for n in [100, 1_000, 10_000, 100_000]:
    # Simulate A/B test
    a = rng.binomial(1, 0.100, n)
    b = rng.binomial(1, 0.101, n)
    # Two-sample z-test approximation using t-test on binary data
    t, p = stats.ttest_ind(b, a)
    sig = "SIGNIFICANT ✓" if p < 0.05 else "not significant"
    print(f"  n={n:>7}: p={p:.4f}  {sig}")

print()
print("With 100,000 users per group, a 0.1pp difference is 'statistically significant'")
print("but the practical value is negligible. Always report effect size alongside p-values.")
print()

# Effect size example
scores_new = rng.normal(loc=73, scale=12, size=50)
mu_base    = 70
d = (scores_new.mean() - mu_base) / scores_new.std(ddof=1)
t_s, p_s = stats.ttest_1samp(scores_new, popmean=mu_base)

print(f"Teaching intervention: sample mean={scores_new.mean():.1f}, baseline={mu_base}")
print(f"t={t_s:.3f},  p={p_s:.4f}")
print(f"Cohen's d = {d:.3f}  ({'small' if abs(d)<0.3 else 'medium' if abs(d)<0.6 else 'large'} effect)")

**The key lesson:** Always interpret statistical significance alongside effect size.

- Small p-value + small effect size → statistically real but practically useless
- Large p-value + large effect size → possibly underpowered (need more data)
- Small p-value + large effect size → what you actually want to see

## Summary

| Concept | Meaning |
|---|---|
| H₀ (null) | Nothing is happening; the default assumption |
| H₁ (alternative) | What you believe is true and want to detect |
| α | Maximum acceptable Type I error rate (usually 0.05) |
| p-value | P(result this extreme or more) if H₀ were true |
| Reject H₀ | p < α — evidence strong enough to dismiss the null |
| Type I error | Rejecting H₀ when it's true (false positive, rate = α) |
| Type II error | Failing to reject H₀ when it's false (false negative, rate = β) |
| Power | 1 − β — probability of correctly detecting a real effect |
| Cohen's d | Effect size — separates statistical from practical significance |

**Decision rule:** p < α → reject H₀. Never say "proved" or "disproved" — always "evidence" and "fail to reject."

Next up: **Hypothesis Testing II** — comparing two groups, testing proportions, and checking assumptions.

## Practice Questions 1

1. A clinic claims their blood pressure medication reduces systolic BP by an average of 10 mmHg. You test it on 20 patients and get a mean reduction of 7.5 mmHg with std 6.2 mmHg. Conduct a one-sample t-test (H₀: reduction = 10, H₁: reduction ≠ 10) at α = 0.05.

2. Using `scipy.stats.ttest_1samp`, verify the result computationally and build a 95% CI. What do you conclude?

3. In the above test: what Type of error would you be making if the medication truly reduces BP by 10 mmHg but your test fails to reject H₀?

## Practice Questions 2

1. A manufacturing line is supposed to produce bolts with diameter exactly 12 mm (std=0.3 mm). You sample 36 bolts and get a mean diameter of 12.08 mm.
   - State H₀ and H₁ for testing whether the machine is miscalibrated
   - Compute the t-statistic manually (show the formula)
   - Use `stats.ttest_1samp` to confirm
   - Compute Cohen's d. Is the effect large or small?

2. A marketing team runs a survey of 500 customers after a new campaign. 65 say they'd buy the product, vs a historical baseline of 10%. Is the improvement statistically significant? (Treat responses as 0/1 and use a one-sample t-test as an approximation.)